In [ ]:
import os
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

def generate_smart_tree(startpath, max_files_to_print=15):
    print(f"\n📂 Scanning directory: {startpath}\n")
    print("="*50)

    for root, dirs, files in os.walk(startpath):
        # Calculate indentation level
        level = root.replace(startpath, '').count(os.sep)
        indent = ' ' * 4 * level
        folder_name = os.path.basename(root)

        # Print the folder name
        if folder_name:
            print(f"{indent}📁 {folder_name}/")

        subindent = ' ' * 4 * (level + 1)

        # If there are too many files, summarize them instead of printing all 85,000
        if len(files) > max_files_to_print:
            ext_counts = {}
            for f in files:
                ext = os.path.splitext(f)[1].lower() or '[no ext]'
                ext_counts[ext] = ext_counts.get(ext, 0) + 1

            summary = ", ".join([f"{count} {ext} files" for ext, count in ext_counts.items()])
            print(f"{subindent}⚠️ [{len(files)} items hidden. Summary: {summary}]")
        else:
            # Print individual files
            for f in files:
                print(f"{subindent}📄 {f}")

# 2. Define your old Thesis folder path
# Note: Google Colab maps "My Drive" to "/content/drive/MyDrive"
target_path = "/content/drive/MyDrive/Thesis_Latrest_Version/Thesis"

# 3. Run the generator
if os.path.exists(target_path):
    generate_smart_tree(target_path)
else:
    print(f"❌ Error: Could not find the folder at '{target_path}'.")
    print("Double check that your Drive is mounted and the folder name is correct.")

Streaming output truncated to the last 5000 lines.
                                    📄 meson.build
                                    📄 setup.py
                                    📁 __pycache__/
                                        📄 setup.cpython-310.pyc
                            📁 __pycache__/
                                ⚠️ [67 items hidden. Summary: 67 .pyc files]
                        📁 __pycache__/
                            ⚠️ [30 items hidden. Summary: 30 .pyc files]
                    📁 _pyinstaller/
                        📄 hook-numpy.py
                        📄 __init__.pyi
                        📄 hook-numpy.pyi
                        📄 __init__.py
                        📁 __pycache__/
                            📄 __init__.cpython-310.pyc
                            📄 hook-numpy.cpython-310.pyc
                        📁 tests/
                            📄 test_pyinstaller.py
                            📄 pyinstaller-smoke.py
                          

In [ ]:
# 1. Mount your Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Check the hardware Google gave us
print("\n--- GPU HARDWARE CHECK ---")
!nvidia-smi

# 3. Verify we can see your Thesis folder
import os
thesis_path = "/content/drive/MyDrive/Thesis_Latrest_Version/Thesis"

print("\n--- FOLDER CHECK ---")
if os.path.exists(thesis_path):
    print("✅ SUCCESS: Thesis folder found!")
else:
    print("❌ ERROR: Cannot find the Thesis folder. Please check the path.")

Mounted at /content/drive

--- GPU HARDWARE CHECK ---
Tue Jun  2 15:55:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  

In [ ]:
!pip install ultralytics

In [ ]:
import os
import sys
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive')

# 2. EXTRACT CUSTOM CODE FROM WINDOWS ENVIRONMENT
print("🔄 Isolating custom YOLO26 library from Windows binaries...")
windows_site_packages = "/content/drive/MyDrive/Thesis_Latrest_Version/Thesis/thesis_env/Lib/site-packages"
custom_ultra_path = os.path.join(windows_site_packages, "ultralytics")
local_ultra_path = "/content/ultralytics"

# Copy ONLY the ultralytics folder to Colab's local Linux storage
if os.path.exists(custom_ultra_path):
    if not os.path.exists(local_ultra_path):
        !cp -r "$custom_ultra_path" /content/
        print("✅ Custom library copied to safe environment!")
    else:
        print("✅ Custom library already exists in safe environment!")
else:
    print(f"❌ FATAL ERROR: Could not find {custom_ultra_path}")
    sys.exit()

# 3. CLEAN UP & HIJACK
!pip uninstall ultralytics -y > /dev/null 2>&1
!rm -rf /usr/local/lib/python3.12/dist-packages/ultralytics

# Point Python to the safe /content folder (NOT the Windows folder)
if windows_site_packages in sys.path:
    sys.path.remove(windows_site_packages)
sys.path.insert(0, '/content')

# 4. VERIFY THE HIJACK
import ultralytics
print(f"🔥 LIBRARY HIJACK SUCCESS! Using YOLO from: {ultralytics.__file__}")

from ultralytics import YOLO

# ==========================================
# 5. DEFINE PATHS
# ==========================================
thesis_root = "/content/drive/MyDrive/Thesis_Latrest_Version/Thesis"
last_one_root = "/content/drive/MyDrive/Thesis_Latrest_Version/Last_One"
os.makedirs(last_one_root, exist_ok=True)
runs_output = os.path.join(last_one_root, "Colab_Runs")
os.makedirs(runs_output, exist_ok=True)
old_weights = os.path.join(thesis_root, "Safe_Backups_HighRes/Dhaka_Combined_1024px_best.pt")

# ==========================================
# 6. GENERATE SAHI DATASET YAML
# ==========================================
sahi_yaml_path = os.path.join(last_one_root, "Dhaka_Traffic_SAHI.yaml")
sahi_yaml_content = f"""
path: {thesis_root}
train: SAHI_Dataset/images/train
val: Master_Dataset_Local/images/val

nc: 8
names:
  0: Bike
  1: Bus/Microbus
  2: Car
  3: CNG
  4: Cycle
  5: People
  6: Rickshaw
  7: Truck
"""
with open(sahi_yaml_path, 'w') as f:
    f.write(sahi_yaml_content)

# ==========================================
# 7. GENERATE YOUR CUSTOM YOLO26-SAHI-PRO
# ==========================================
pro_yaml_path = os.path.join(last_one_root, "yolo26m-sahi-pro.yaml")
pro_yaml_content = """
# YOLO26m-Pro: Custom CBAM/DCFM + P2 Small Object Head
nc: 8
scales:
  m: [0.67, 0.75, 1024]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, DCFM, [192, 192]]  # YOUR CUSTOM DCFM
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, CBAM, [384]]       # YOUR CUSTOM CBAM
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, SPPF, [512, 5]]
  - [-1, 2, C2PSA, [512]]

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 8], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 5], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, False]]

  # --- P2 SMALL OBJECT RECOVERY ---
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 3], 1, Concat, [1]]
  - [-1, 2, C3k2, [128, False]] # 21 (P2 Head Output)

  - [-1, 1, Conv, [128, 3, 2]]
  - [[-1, 18], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, False]] # 24 (P3 Head Output)

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 15], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]] # 27 (P4 Head Output)

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, True]]  # 30 (P5 Head Output)

  - [[21, 24, 27, 30], 1, Detect, [nc]]
"""
with open(pro_yaml_path, 'w') as f:
    f.write(pro_yaml_content)

print("✅ Custom Architecture Built Successfully!")

# ==========================================
# 8. INITIATE 3-EPOCH DIAGNOSTIC TEST
# ==========================================
print("🚀 INITIATING YOLO26-SAHI-PRO DIAGNOSTIC...")

model = YOLO(pro_yaml_path)

print(f"🧠 Transferring knowledge from: {old_weights}")
model.load(old_weights)

model.train(
    data=sahi_yaml_path,
    epochs=3,
    batch=16,
    imgsz=640,
    device=0,
    project=runs_output,
    name="Dhaka_SAHI_Pro_Diagnostic",
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔄 Isolating custom YOLO26 library from Windows binaries...
✅ Custom library copied to safe environment!


AttributeError: module 'cv2' has no attribute 'imshow'